In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
def is_full_load(spark, schema, target_table_name, target_table_path):
    try:
        spark_tables = spark.catalog.listTables(schema)
    except Exception:
        spark_tables = []

    is_target_table_exist = any(
        x.database == schema.lower() and x.name == target_table_name.lower() for x in spark_tables
    )

    if not is_target_table_exist:
        return True

    is_delta_table = DeltaTable.isDeltaTable(spark, target_table_path)

    if is_delta_table:
        try:
            is_empty = spark.read.format("delta").load(target_table_path).isEmpty()
            return is_empty
        except Exception:
            return True
    else:
        return True

In [0]:
def add_dataframe_metadata(df):
    """
    Add metadata columns to a dataframe
    """
    df = df.withColumn("ingest_file_name", df["_metadata.file_path"])\
    .withColumn("file_modification_time", df["_metadata.file_modification_time"])

    return df
